# TrkA_88 원본 AF2 코드 재예측 · v0.4.0

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_protein/blob/main/notebooks/01_af2_filter_T4.ipynb?v=0.4.0)

T4에서 전체 실행하면 논문에 연결된 nrbennet/dl_binder_design의 predict.py를 직접 실행합니다. 원 저장소의 모델, 특징 전처리, 표적 템플릿, 초기 구조, 사슬 경계 처리와 recycle 3회 설정을 사용합니다. 원 코드 커밋은 cafa3853ac94dceb1b908c8d9e6954d71749871a로 고정합니다.

공개 RFdiffusion 뼈대의 첫 사슬에 공개 TrkA_88 서열을 PyRosetta로 넣습니다. 주사슬 좌표가 유지되는지 확인한 뒤 원 코드에 전달합니다. PyRosetta와 원본 AF2용 환경을 설치하며, 같은 Colab 세션의 AF2 가중치를 재사용합니다.

결과 표는 TrkA를 고정한 결합체 위치·방향 오차를 보여줍니다. 논문 보고값 1.4 Å, 공개 AF2 구조의 재계산값 약 1.39 Å, 이번 예측값을 비교합니다. 원 코드의 신뢰도 점수와 PDB, 입력, 설정, 로그를 ZIP 하나에 저장합니다.

[원본 AF2 실행 코드](https://github.com/nrbennet/dl_binder_design/blob/cafa3853ac94dceb1b908c8d9e6954d71749871a/af2_initial_guess/predict.py) · [논문 그림 6](https://www.nature.com/articles/s41586-023-06415-8) · [공개 자료](https://doi.org/10.6084/m9.figshare.22662094.v1)


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import subprocess
import sys

checkout = Path(mkdtemp(prefix="trka88_main_", dir="/content")) / "repo"
clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--single-branch", "--branch", "main",
     "https://github.com/HisameOgasahara/tmp_protein.git", str(checkout)],
    capture_output=True, text=True,
)
if clone.returncode:
    print(clone.stderr)
    clone.check_returncode()
revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
print(f"main: {revision[:12]}")


In [ ]:
import json
import pandas as pd
from IPython.display import display

log = checkout.parent / "trka88_workflow.log"
result_zip = None
with log.open("w", encoding="utf-8") as handle:
    with subprocess.Popen(
        [sys.executable, "-u", str(checkout / "src/af2_filter/workflow.py"), revision],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
    ) as process:
        for line in process.stdout:
            handle.write(line)
            handle.flush()
            print(line, end="", flush=True)
            if line.strip().endswith(".zip") and not line.startswith("Failure archive:"):
                result_zip = Path(line.strip())
        if process.wait():
            raise RuntimeError(f"AF2 failed. Full log: {log}")
if result_zip is None or not result_zip.is_file():
    raise FileNotFoundError(f"Missing result ZIP. Full log: {log}")
report = json.loads((result_zip.with_suffix("") / "comparison.json").read_text())
display(pd.DataFrame([
    {"비교": "논문 보고값", "표적 기준 결합체 RMSD (Å)": report["paper_pose_rmsd_angstrom"]},
    {"비교": "공개 AF2 vs 설계", "표적 기준 결합체 RMSD (Å)": report["published_af2_vs_design_angstrom"]},
    {"비교": "새 AF2 vs 설계", "표적 기준 결합체 RMSD (Å)": report["reprediction_vs_design_angstrom"]},
    {"비교": "새 AF2 vs 공개 AF2", "표적 기준 결합체 RMSD (Å)": report["reprediction_vs_published_af2_angstrom"]},
]).round(3))
print(report["original_af2_scores"])


In [ ]:
from google.colab import files
files.download(str(result_zip))
